# 01. Data Preprocessing and STRING Network Mapping

# Overview

This notebook performs the preprocessing and network construction steps used in the study **"Network-Based Annotation Enrichment to Identify Functional Subclusters in Breast Cancer using NetAN."**

# Objectives

- Load the curated breast cancer-associated gene set.
- Load the STRING protein–protein interaction (PPI) network.
- Map breast cancer-associated genes to the STRING network.
- Construct the breast cancer protein interaction network.
- Export the mapped network for Cytoscape visualization.
- Generate input files for downstream NetAN analyses.

# Input

- Curated breast cancer gene set
- STRING v12.0 protein–protein interaction network

# Output

- Mapped breast cancer gene list
- Breast cancer interaction network
- Cytoscape node and edge files
- Processed datasets for downstream enrichment analyses

# Import Required Libraries

The following Python libraries are used for data handling, network analysis, and visualization throughout the preprocessing workflow.

In [3]:
# Standard libraries
import os
import sys
from pathlib import Path

# Data analysis
import numpy as np
import pandas as pd

# Network analysis
import networkx as nx

# Add NetAN utilities to the Python path
sys.path.append(".")

# Import NetAN functions
from utilities.nw_utils import load_nw, map2nw

# Load the STRING Protein–Protein Interaction Network

The STRING-derived protein–protein interaction (PPI) network serves as the reference interaction network for all downstream analyses. Breast cancer-associated genes are mapped onto this network before enrichment and clustering analyses are performed.

In [4]:
# Load the STRING interaction network
network = load_nw("STRINGdb")

# Display the first few rows
network.head()

,p2,score
p1,,
ARF5,PDE1C,0.155155
ERCC1,PDE1C,0.255255
TLL1,PDE1C,0.153153
PRSS22,PDE1C,0.205205
LAPTM4A,PDE1C,0.175175


# Load the Curated Breast Cancer Gene Set

The curated breast cancer-associated gene set compiled for this study serves as the input for all downstream analyses. Before network construction, the gene list is imported, inspected, and prepared for mapping to the STRING protein–protein interaction network.

In [7]:
from pathlib import Path

# Create the output directory if it does not already exist
output_dir = Path("results")
output_dir.mkdir(parents=True, exist_ok=True)

# Path to the curated breast cancer gene set
gene_file = Path("data/raw/unique_genes_clean.csv")

# Load the gene list
genes = (
    pd.read_csv(gene_file, header=None)
      .iloc[:, 0]
      .dropna()
      .astype(str)
      .str.strip()
      .tolist()
)

print("=== Breast Cancer Gene Set ===")
print(f"Total genes loaded: {len(genes)}")
print(f"First 10 genes: {genes[:10]}")

=== Breast Cancer Gene Set ===
Total genes loaded: 997
First 10 genes: ['AARSD1', 'AASDHPPT', 'AATF', 'ABCB6', 'ABCC3', 'ABI3', 'ACACA', 'ACADVL', 'ACAP1', 'ACAT1']


# Map the Gene Set to the STRING Protein–Protein Interaction Network

The curated breast cancer-associated genes are mapped to the STRING interaction network using NetAN. Genes successfully mapped to the network are retained for downstream network construction, enrichment analysis, and clustering, while unmapped genes are recorded separately for transparency and reproducibility.

In [8]:
# Map genes to the STRING interaction network
expanded_genes, report = map2nw(
    start_nodes=np.array(genes),
    nw=network,
    augment_factor=0
)

mapped_genes = sorted(set(genes) - set(report["missing_genes"]))
unmapped_genes = sorted(
    set(report["missing_genes"]).union(report["no_IDs"])
)

print("=== Mapping Summary ===")
print(f"Mapped genes: {len(mapped_genes)}")
print(f"Unmapped genes: {len(unmapped_genes)}")

288 out of 997 genes not found.
35 out of 997 input IDs were mapped to STRING IDs and alternative IDs.
253 out of 997 have not been found with STRING IDs or alternative IDs.
0 genes added.
=== Mapping Summary ===
Mapped genes: 709
Unmapped genes: 288


In [11]:
# Save mapping results
pd.DataFrame({"gene": mapped_genes}).to_csv(
    output_dir / "mapped_genes.csv",
    index=False
)

pd.DataFrame({"gene": unmapped_genes}).to_csv(
    output_dir / "unmapped_genes.csv",
    index=False
)

print("Mapping results saved successfully.")
# Large reference databases

Mapping results saved successfully.
